# Module 10.1: Quantization Fundamentals

Welcome to Module 10! You've built a full Transformer, but as you scale it up, you hit a brutal reality: **VRAM (Video RAM) is expensive and scarce.**

A 70 Billion parameter model in 16-bit precision requires ~140GB of VRAM just to load the weights (let alone context and activations!). 
How can we run these models on consumer GPUs? The answer is **Quantization**: shrinking the neural network's parameters by squeezing high-precision floating-point numbers into low-precision integer formats (like 8-bit or 4-bit).

## 1. Absolute Maximum (Absmax) Quantization

### The Concept
A standard `float16` or `float32` number can represent highly precise fractional values over a massive range. An `int8` (8-bit integer) can only represent 256 discrete whole numbers: from `-128` to `127`.

To squeeze our weights into `int8`, we need a mapping strategy. **Absmax Quantization** is symmetric. We find the absolute maximum value in our weight tensor, say `3.2`, and we define that as our maximum integer `127`. Everything else is scaled proportionally.

### The Math
$$ Scale = \frac{127}{max(|W|)} $$
$$ W_{quantized} = round(W \times Scale) $$
$$ W_{dequantized} = rac{W_{quantized}}{Scale} $$

In [1]:
import torch
import matplotlib.pyplot as plt

def absmax_quantize(tensor):
    # 1. Find the absolute maximum value
    absmax = torch.max(torch.abs(tensor))
    
    # 2. Calculate the scaling factor
    # 127 is the max value positive value an int8 can hold
    scale = 127.0 / absmax
    
    # 3. Scale the tensor and round to nearest integer
    quantized = torch.round(tensor * scale)
    
    # 4. Cast safely to int8 (clamping just to be extremely safe against rounding errors)
    quantized_int8 = torch.clamp(quantized, -128, 127).to(torch.int8)
    return quantized_int8, scale

def absmax_dequantize(quantized_tensor, scale):
    # Cast back to float32 before dividing, and divide by the exact same scale
    return quantized_tensor.to(torch.float32) / scale

## 2. Let's see the Loss of Precision

Quantization is a "lossy" compression. By forcing floating point numbers into integer buckets, we lose the tiny fractional details. Let's visualize this.

In [2]:
# Create a mock weight matrix with standard normal distribution (like real neural weights)
original_weights = torch.randn(10, 10)

quantized_weights, scale = absmax_quantize(original_weights)
recovered_weights = absmax_dequantize(quantized_weights, scale)

print(f"Scale Factor: {scale:.4f}")
print(f"Original Byte Size: {original_weights.element_size() * original_weights.nelement()} bytes as Float32")
print(f"Quantized Byte Size: {quantized_weights.element_size() * quantized_weights.nelement()} bytes as Int8 (We saved 75% memory!)")

# Calculate the Error (Mean Squared Error)
mse_error = torch.nn.functional.mse_loss(original_weights, recovered_weights)
print(f"\nPrecision Loss (MSE): {mse_error.item():.6f}")

# Look at a single specific weight to understand rounding error
print(f"\nOriginal weight value: {original_weights[0, 0].item():.4f}")
print(f"Recovered weight value: {recovered_weights[0, 0].item():.4f}")

## 3. Zero-Point (Asymmetric) Quantization

### The Problem with Absmax
Absmax assumes your neural network weights are perfectly symmetric around zero. But what if all your weights are between `10.0` and `20.0`? If you use Absmax, the max is `20.0`, mapped to `127`. The value `-127` would represent `-20.0`. But you have NO negative weights! Half of your precious `int8` buckets (`-128` to `0`) are completely wasted.

### The Solution: Zero-Point
Instead of mapping the max absolute value, we find the overall `min` and `max` of our tensor, and map that exact range to `-128` and `127` (or `0` to `255` for uint8). We calculate a `zero_point` that shifts the numbers.

In [3]:
def asymmetric_quantize(tensor):
    t_min, t_max = tensor.min(), tensor.max()
    
    # Now we map the range (max - min) to 255 (the range of uint8)
    scale = 255.0 / (t_max - t_min)
    
    # Calculate the zero point (the integer value that represents 0.0 float)
    zero_point = torch.round(-t_min * scale).to(torch.int32)
    
    # Quantize, shift, and clip
    quantized = torch.clamp(torch.round(tensor * scale) + zero_point, 0, 255).to(torch.uint8)
    
    return quantized, scale, zero_point

def asymmetric_dequantize(quantized_tensor, scale, zero_point):
    # Order of operations matters: Shift back, then divide by scale
    return (quantized_tensor.to(torch.float32) - zero_point) / scale

# Test with purely positive weights (where Absmax is inefficient)
positive_weights = torch.rand(10, 10) * 10 + 10  # Random floats from 10.0 to 20.0

q_asym, scale_asym, zp_asym = asymmetric_quantize(positive_weights)
recovered_asym = asymmetric_dequantize(q_asym, scale_asym, zp_asym)

print(f"Precision Loss for highly skewed weights (MSE): {torch.nn.functional.mse_loss(positive_weights, recovered_asym).item():.6f}")

## 4. Modern Quantization (LLM.int8(), GPTQ, AWQ)

If it's this simple, why are there so many papers?

- **Outlier Features**: In massive LLMs, some hidden dimensions suddenly spike to huge values (e.g., `100.0`). If you quantize the whole matrix, that single `100.0` will force the `scale` so low that all normal weights get crushed into the `int8` value `0`. Papers like **LLM.int8()** solve this by keeping outliers in `float16` and quantizing the rest.
- **Group Quantization**: Instead of finding one `Scale` for the whole matrix, we find a separate `Scale` for every chunk of 128 numbers. This retains vastly more precision.
- **4-Bit**: People have successfully pushed weights down to 4 bits using Non-Linear Quantization (like NF4 in QLoRA), drastically reducing VRAM usage further.